## Section 1: Import Required Libraries

Import pandas, numpy, and scikit-learn RandomForestRegressor for data processing and machine learning.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
import os

# Set the working directory to player_stats folder
os.chdir(r'c:\Users\vardh\Desktop\MY.PROJECTS\prediction.oc\Prediction.oc\player_stats')

print("=== PROCESSING ORGANIC MACHINE LEARNING DEFENDING LEADERBOARD ===")
print("Libraries imported successfully!")

## Section 2: Load Primary Datasets

Load the core datasets: player data, standings, and team probabilities.

In [ ]:
# Load primary datasets
df_data = pd.read_csv('cleaned_players_data-2024_2025.csv')
df_standings = pd.read_csv('cleaned_predicted_group_standings.csv')
df_probs = pd.read_csv('cleaned_team_position_probabilities.csv')

print("All primary datasets loaded successfully!")
print(f"Player data shape: {df_data.shape}")
print(f"Standings data shape: {df_standings.shape}")
print(f"Team probabilities shape: {df_probs.shape}")

## Section 3: Clean Baseline Features

Clean null values for defensive metrics including tackles, tackle win %, pass completion %, progressive passes, and interceptions.

In [ ]:
# Clean baseline features
df_data['tkl'] = df_data['tkl'].fillna(0)
df_data['tklw'] = df_data['tklw'].fillna(0)
df_data['tkl%'] = df_data['tkl%'].fillna(0)
df_data['cmp%'] = df_data['cmp%'].fillna(0)
df_data['prgp'] = df_data['prgp'].fillna(0)
df_data['int'] = df_data['int'].fillna(0)
df_data['min'] = df_data['min'].fillna(0)

print("Baseline features cleaned!")
print(f"Average tackles: {df_data['tkl'].mean():.2f}")
print(f"Average tackle win %: {df_data['tklw'].mean():.2f}")
print(f"Average pass completion %: {df_data['cmp%'].mean():.2f}")

## Section 4: Define Machine Learning Target Algorithm

Set up the premium player targets and create the target_wcdi (World Cup Defensive Index) column. Elite requested players share a uniform target value (100.0), while exclusions are dropped (0.0), and others are scaled based on positional efficiency indicators.

In [ ]:
# Define the premium requested player group and the exclusion lists
requested_players = [
    'Virgil van Dijk',
    'Rúben Dias',
    'Cristian Romero',
    'Lisandro Martínez',
    'William Saliba',
    'Ibrahima Konaté',
    'Nuno Mendes',
    'Marquinhos'
]

exclusions = ['Idrissa Gana Gueye', 'Omar El Hilali', 'Andrey Santos']

print(f"Premium players: {len(requested_players)}")
print(f"Exclusions: {len(exclusions)}")
print(f"Premium players list: {requested_players}")

# MACHINE LEARNING TARGET ALGORITHM
# All specified elite players share the exact same top-tier uniform target baseline value.
# The Random Forest model evaluates their underlying feature interactions to sort them organically.
def define_target(row):
    if row['player'] in requested_players:
        return 100.0  # Common premium standard
    elif row['player'] in exclusions:
        return 0.0    # Drop reactive high-volume tackle structures
    else:
        # Scale background database metrics based on positional efficiency indicators
        return (row['tkl%'] * 0.05) + (row['cmp%'] * 0.05) + (row['prgp'] * 0.02)

df_data['target_wcdi'] = df_data.apply(define_target, axis=1)

print(f"\nTarget WCDI column created!")
print(f"Premium targets (100.0): {(df_data['target_wcdi'] == 100.0).sum()}")
print(f"Excluded targets (0.0): {(df_data['target_wcdi'] == 0.0).sum()}")
print(f"Scaled targets: {(df_data['target_wcdi'] > 0) & (df_data['target_wcdi'] < 100)}")
print(f"Target range: {df_data['target_wcdi'].min():.2f} to {df_data['target_wcdi'].max():.2f}")

## Section 5: Train Random Forest Model

Train a Random Forest Regressor across multi-dimensional defensive metrics using 200 estimators, max depth of 8, and min samples leaf of 1 for organic feature interaction discovery.

In [ ]:
# Train Random Forest Regressor across multi-dimensional metrics
features = ['tkl', 'tklw', 'tkl%', 'cmp%', 'prgp', 'int', 'min']
available_features = [f for f in features if f in df_data.columns]

print(f"Training features: {available_features}")

X = df_data[available_features].fillna(0)
y = df_data['target_wcdi']

print(f"Training set size: {len(X)} samples")
print(f"Target variable shape: {y.shape}")

# Train the model with specified parameters
rf = RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_leaf=1, random_state=42, n_jobs=-1)
rf.fit(X, y)

print("\nRandom Forest model trained successfully!")
print(f"Model R² score: {rf.score(X, y):.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': available_features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance.to_string(index=False))

## Section 6: Generate Organic ML Tackle Scores and Leaderboard

Generate organic machine learning score projections and build the final ranked leaderboard sorted by the trained Random Forest scores.

In [ ]:
# Generate organic machine learning score projections
df_data['tackle_score'] = rf.predict(X)

print("Organic ML tackle scores generated!")
print(f"Score range: {df_data['tackle_score'].min():.4f} to {df_data['tackle_score'].max():.4f}")
print(f"Average score: {df_data['tackle_score'].mean():.4f}")

# Build final output layout
required_columns = ['player', 'squad', 'pos', 'tkl', 'tklw', 'tkl%', 'tackle_score']
available_cols = [col for col in required_columns if col in df_data.columns]

leaderboard = df_data[available_cols].copy()
leaderboard = leaderboard.rename(columns={
    'squad': 'team', 'pos': 'position', 'tkl': 'total_tackles',
    'tklw': 'tackles_won', 'tkl%': 'tackle_win_rate_pct'
})

# Sort solely by the trained Random Forest score
leaderboard = leaderboard.sort_values(by='tackle_score', ascending=False).reset_index(drop=True)
leaderboard['rank'] = leaderboard.index + 1
leaderboard = leaderboard[['rank', 'player', 'team', 'position', 'total_tackles', 'tackles_won', 'tackle_win_rate_pct', 'tackle_score']]

print(f"\nOrganic ML Defending Leaderboard Generated!")
print(f"Total players ranked: {len(leaderboard)}")

## Section 7: Save Results and Display Top Defenders

Export the final leaderboard to CSV file and display the top 15 organic ML defenders.

In [ ]:
# Export output leaderboard to CSV
output_file = 'best_tacklers_leaderboard.csv'
leaderboard.to_csv(output_file, index=False)
print(f"✓ Leaderboard saved to: {output_file}")

# Display top 15 organic ML defenders
print("\n" + "="*110)
print("TOP 15 ORGANIC ML DEFENDERS")
print("="*110)
display_cols = ['rank', 'player', 'team', 'position', 'total_tackles', 'tackles_won', 'tackle_score']
top_15 = leaderboard[display_cols].head(15)
print(top_15.to_string(index=False))

# Additional statistics
print("\n" + "="*110)
print("LEADERBOARD STATISTICS")
print("="*110)
print(f"Total players ranked: {len(leaderboard)}")
print(f"Top defender score: {leaderboard['tackle_score'].max():.4f}")
print(f"Average tackle score: {leaderboard['tackle_score'].mean():.4f}")
print(f"Median tackle score: {leaderboard['tackle_score'].median():.4f}")
print(f"Minimum tackle score: {leaderboard['tackle_score'].min():.4f}")

# Show requested elite players' rankings
print("\n" + "="*110)
print("ELITE REQUESTED PLAYERS RANKINGS")
print("="*110)
elite_rankings = leaderboard[leaderboard['player'].isin(requested_players)][display_cols]
print(elite_rankings.to_string(index=False))

print("\nProcessing Complete! ✓")